# CPU screening execution snapshot

Only CPU reference/screening calls ran here. GRU calls and subsequent stages are deferred to the GPU notebook. The canonical notebook source and data were not changed; the reference cell below omits the deferred GRU call explicitly.

In [1]:
import os
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
import gc, hashlib, importlib.metadata as metadata, json, time, copy, random, threading
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

RUN_EXPERIMENTS=os.environ.get('PA3_RUN_V003','0')=='1'
RESUME=os.environ.get('PA3_RESUME_V003','0')=='1'
PROJECT=Path.cwd().resolve()
if not (PROJECT/'data').is_dir() and (PROJECT/'ventilator-pressure-prediction/data').is_dir():
    PROJECT=PROJECT/'ventilator-pressure-prediction'
OUT=PROJECT/'outputs/controlled_v003'
PREV=PROJECT/'outputs/tuning_v002'
BASE=PROJECT/'outputs/baseline_v001'
DATA=PROJECT/'data'
MODEL_SEEDS=[20261102,20261103,20261104]
B_SPLIT_SEED=20261204
THREADS=4
MIN_GAIN=1e-6
print('Mode:', 'run controlled experiments' if RUN_EXPERIMENTS else 'review saved results')
print('Output:', OUT)
if RUN_EXPERIMENTS:
    import lightgbm as lgb
    from sklearn.model_selection import train_test_split

Mode: run controlled experiments
Output: /home/v2gao/dsc102/pa3/gym-workspace/ventilator-pressure-prediction/outputs/controlled_v003


In [2]:
STEPS = 80
DTYPES = {
    "id": "int64", "breath_id": "int32", "R": "int8", "C": "int8",
    "time_step": "float32", "u_in": "float32", "u_out": "int8",
    "pressure": "float32",
}


def digest(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

def features(frame: pd.DataFrame) -> tuple[np.ndarray, list[str], np.ndarray]:
    """Return features in original row order, after verifying 80-row blocks."""
    if len(frame) % STEPS:
        raise ValueError("Unexpected breath length; do not reshape this file")
    blocks = frame["breath_id"].to_numpy().reshape(-1, STEPS)
    breath_ids = blocks[:, 0]
    if not np.all(blocks == breath_ids[:, None]):
        raise ValueError("Breaths are not contiguous 80-row blocks")
    if len(np.unique(breath_ids)) != len(breath_ids):
        raise ValueError("Repeated breath blocks")
    if not frame["id"].is_unique:
        raise ValueError("Duplicate row IDs")
    u = frame["u_in"].to_numpy().reshape(-1, STEPS)
    t = frame["time_step"].to_numpy().reshape(-1, STEPS)
    if not np.all(np.diff(t, axis=1) > 0):
        raise ValueError("Timestamps must increase inside each breath")
    for column in ("R", "C"):
        values = frame[column].to_numpy().reshape(-1, STEPS)
        if not np.all(values == values[:, :1]):
            raise ValueError(f"{column} changes within a breath")

    columns = {}
    for column in ("R", "C", "time_step", "u_in", "u_out"):
        columns[column] = frame[column].to_numpy(dtype=np.float32)
    step = np.arange(STEPS)
    dt = np.diff(t, axis=1, prepend=t[:, :1])
    cumulative = np.cumsum(u, axis=1, dtype=np.float32)
    columns["step"] = np.broadcast_to(step, u.shape).ravel()
    columns["dt"] = dt.ravel()
    columns["control_sum"] = cumulative.ravel()
    columns["control_mean"] = (cumulative / (step + 1)).ravel()
    # Valve opening is not measured flow: this is a history proxy, not volume.
    columns["control_integral"] = np.cumsum(u * dt, axis=1).ravel()
    for lag in (1, 2, 4, 8):
        shifted = np.empty_like(u)
        shifted[:, :lag] = u[:, :1]
        shifted[:, lag:] = u[:, :-lag]
        columns[f"u_in_lag_{lag}"] = shifted.ravel()
        columns[f"u_in_change_{lag}"] = (u - shifted).ravel()
    padded_sum = np.concatenate([np.zeros((len(u), 1), dtype=np.float32), cumulative], axis=1)
    for window in (4, 8):
        begin = np.maximum(0, step + 1 - window)
        columns[f"u_in_mean_{window}"] = (
            (padded_sum[:, step + 1] - padded_sum[:, begin]) / (step + 1 - begin)
        ).ravel()
    names = list(columns)
    matrix = np.column_stack(list(columns.values())).astype(np.float32, copy=False)
    if not np.isfinite(matrix).all() or "pressure" in names:
        raise ValueError("Invalid features")
    return matrix, names, breath_ids

In [3]:
TREE_PARAMS=dict(n_estimators=2500,num_leaves=255,learning_rate=.04,
                 min_child_samples=20,max_bin=255,reg_lambda=2.)
GRU_PARAMS=dict(hidden=128,layers=2,dropout=.15,lr=.00075,batch_size=256,
                max_epochs=100,patience=8,max_seconds=600,min_lr=0.)
def spec(family='LightGBM',params=None,history='both',rc='numeric'):
    return dict(family=family,params=copy.deepcopy(params if params is not None else TREE_PARAMS),
                history=history,rc=rc)
TREE_REFERENCE=spec()
GRU_REFERENCE=spec('GRU',GRU_PARAMS)
TREE_SCREEN=[
 ('history_neither',spec(history='neither')),
 ('history_lags',spec(history='lags')),
 ('history_changes',spec(history='changes')),
 ('rc_categorical',spec(rc='categorical')),
 ('lr_002',spec(params=dict(TREE_PARAMS,learning_rate=.02))),
 ('lr_008',spec(params=dict(TREE_PARAMS,learning_rate=.08))),
 ('trees_5000',spec(params=dict(TREE_PARAMS,n_estimators=5000))),
 ('leaves_127',spec(params=dict(TREE_PARAMS,num_leaves=127))),
 ('leaves_511',spec(params=dict(TREE_PARAMS,num_leaves=511))),
 ('l2_10',spec(params=dict(TREE_PARAMS,reg_lambda=10.))),
]
GRU_FLOOR=dict(GRU_PARAMS,min_lr=1e-5)
GRU_SCREEN=[
 ('gru_lr_floor',spec('GRU',GRU_FLOOR)),
 ('gru_lr_0003',spec('GRU',dict(GRU_FLOOR,lr=.0003))),
 ('gru_lr_0015',spec('GRU',dict(GRU_FLOOR,lr=.0015))),
 ('gru_width_64',spec('GRU',dict(GRU_FLOOR,hidden=64))),
 ('gru_dropout_030',spec('GRU',dict(GRU_FLOOR,dropout=.30))),
]
PLAN=dict(version=3,model_seeds=MODEL_SEEDS,B_split_seed=B_SPLIT_SEED,minimum_MAE_gain=MIN_GAIN,
    tree_reference=TREE_REFERENCE,gru_reference=GRU_REFERENCE,
    tree_screen=TREE_SCREEN,gru_screen=GRU_SCREEN,
    expansion='Top two non-reference tree arms and best GRU, chosen only by A screening MAE',
    ensemble='Three identical configurations; different model seeds; arithmetic mean; no weight tuning',
    final_rule='Incumbent always eligible. Best A individual must beat incumbent on A and B. Equal ensemble must beat best A individual on A and B and incumbent on B. Choose lowest A MAE among eligible options; no B hyperparameter retuning',
    confirmation='B is drawn from non-A pool; A remains excluded from every fit; B models freshly refit without B feedback, using A-frozen tree counts or GRU epochs and LR schedules',
    loss='Inspiratory L1 for every new fit; native-unit inspiratory MAE for all scoring')
IO_LOCK=threading.RLock()
def json_write(path,value):
    with IO_LOCK:
        temp=path.with_name(path.name+'.tmp')
        temp.write_text(json.dumps(value,indent=2)+'\n')
        os.replace(temp,path)
def append_event(event,**values):
    with IO_LOCK:
        with (OUT/'events.jsonl').open('a') as f:
            f.write(json.dumps(dict(timestamp_utc=datetime.now(timezone.utc).isoformat(),event=event,**values))+'\n')

if RUN_EXPERIMENTS:
    OUT.mkdir(parents=True,exist_ok=RESUME)
    quality=json.loads((PROJECT/'quality_audit_v002.json').read_text())
    assert digest(DATA/'train.csv')==quality['files']['train.csv']['sha256']
    frame=pd.read_csv(DATA/'train.csv',dtype=DTYPES)
    flat,names,breath_ids=features(frame)
    X=flat.reshape(-1,80,len(names)); del flat
    Y=frame.pressure.to_numpy().reshape(-1,80)
    UOUT=frame.u_out.to_numpy().reshape(-1,80)
    RC=frame[['R','C']].to_numpy()[::80]
    ROW_IDS=frame.id.to_numpy().reshape(-1,80)
    all_idx=np.arange(len(breath_ids))
    saved=np.load(BASE/'split_breath_ids.npz')
    ix=pd.Index(breath_ids)
    a_val=np.sort(ix.get_indexer(saved['validation']))
    screen_fit=np.sort(ix.get_indexer(saved['trained']))
    a_fit=np.setdiff1d(all_idx,a_val)
    strata=np.array([f'{r}_{c}' for r,c in RC])
    b_fit,b_val=train_test_split(a_fit,test_size=len(a_val),stratify=strata[a_fit],random_state=B_SPLIT_SEED)
    b_fit=np.sort(b_fit);b_val=np.sort(b_val)
    FITS={'screen':screen_fit,'full':a_fit,'confirmation':b_fit}
    VALS={'A':a_val,'B':b_val}
    assert len(a_fit)==61114 and len(a_val)==6791 and len(screen_fit)==20000
    assert len(b_fit)==54323 and len(b_val)==6791
    for f in FITS.values():
        assert (f>=0).all() and not np.intersect1d(f,a_val).size
    assert not np.intersect1d(b_fit,b_val).size
    assert set(np.unique(RC[:,0]))=={5,20,50} and set(np.unique(RC[:,1]))=={10,20,50}
    source_notebook=json.loads((PROJECT/'controlled_improvements_v003.ipynb').read_text())
    training_source_sha256=hashlib.sha256('\n'.join(''.join(c['source']) for c in source_notebook['cells'] if c['cell_type']=='code').encode()).hexdigest()
    fingerprint=dict(plan=PLAN,features=names,training_source_sha256=training_source_sha256,train_sha256=digest(DATA/'train.csv'),
        original_split_sha256=digest(BASE/'split_breath_ids.npz'),
        groups={k:hashlib.sha256(breath_ids[v].tobytes()).hexdigest() for k,v in dict(screen=screen_fit,Afit=a_fit,Aval=a_val,Bfit=b_fit,Bval=b_val).items()})
    fingerprint=json.loads(json.dumps(fingerprint))
    if RESUME:
        assert json.loads((OUT/'design.json').read_text())==fingerprint, 'Plan or data changed; use a fresh run'
    else:
        json_write(OUT/'design.json',fingerprint)
        np.savez_compressed(OUT/'split_breath_ids.npz',screen=breath_ids[screen_fit],A_fit=breath_ids[a_fit],A_validation=breath_ids[a_val],B_fit=breath_ids[b_fit],B_validation=breath_ids[b_val])
        json_write(OUT/'versions.json',{p:metadata.version(p) for p in ['lightgbm','torch','numpy','pandas','scikit-learn']})
    for split,ids in VALS.items(): np.save(OUT/f'{split}_validation_row_ids.npy',ROW_IDS[ids][UOUT[ids]==0])
    append_event('initialization',resume=RESUME,group_sizes={k:len(v) for k,v in FITS.items()})
    del frame;gc.collect()
    print('Locked group sizes:',{k:len(v) for k,v in FITS.items()},{k:len(v) for k,v in VALS.items()})

Locked group sizes: {'screen': 20000, 'full': 61114, 'confirmation': 54323} {'A': 6791, 'B': 6791}


In [4]:
def transform(values,definition):
    mode=definition['history']
    keep=[i for i,n in enumerate(names)
          if not (n.startswith('u_in_lag_') and mode not in ('both','lags'))
          and not (n.startswith('u_in_change_') and mode not in ('both','changes'))]
    z=np.ascontiguousarray(values[...,keep]);used=[names[i] for i in keep]
    if definition['rc']=='categorical':
        assert definition['family']=='LightGBM'
        for column,levels in [('R',[5,20,50]),('C',[10,20,50])]:
            j=used.index(column);raw=z[...,j].copy()
            assert np.isin(raw,levels).all(), 'Unknown R/C level'
            for i,level in enumerate(levels): z[...,j][raw==level]=i
    return z,used

def flat_scored(indices,definition):
    z,used=transform(X[indices],definition)
    mask=UOUT[indices]==0
    return np.ascontiguousarray(z[mask]),Y[indices][mask],used

def train_diagnostic_indices(indices):
    return np.sort(np.random.default_rng(20261110).choice(indices,min(2048,len(indices)),replace=False))

if RUN_EXPERIMENTS:
    # Verify the factorial removes exactly the intended columns.
    actual={m:transform(X[:1],spec(history=m))[1] for m in ['neither','lags','changes','both']}
    assert [len(actual[m]) for m in ['neither','lags','changes','both']]==[12,16,16,20]
    for lag in [1,2,4,8]:
        assert np.allclose(X[:2,:,names.index('u_in_change_'+str(lag))],
            X[:2,:,names.index('u_in')]-X[:2,:,names.index('u_in_lag_'+str(lag))])
    cat,_=transform(X[:2],spec(rc='categorical'))
    assert np.isin(cat[...,:2],[0,1,2]).all()
    # Perturb later inputs in one breath and require earlier feature rows unchanged.
    sample=pd.read_csv(DATA/'train.csv',nrows=160,dtype=DTYPES)
    orig=features(sample)[0];perturbed=sample.copy()
    perturbed.loc[40:79,'u_in']+=3
    altered=features(perturbed)[0]
    assert np.array_equal(orig[:40],altered[:40]) and np.array_equal(orig[80:],altered[80:])
    append_event('feature_checks_passed',factorial_feature_counts={m:len(v) for m,v in actual.items()})

In [5]:
if RUN_EXPERIMENTS:
    # Notebook-ready GRU helpers. No code below executes training until fit_gru_trial is called.
    # Set this before the first CUDA operation in a fresh notebook kernel.
    import os
    os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')

    import copy
    import json
    import random
    import time
    from pathlib import Path

    import numpy as np
    import torch
    from torch import nn
    from torch.utils.data import DataLoader, TensorDataset

    GRU_CONFIGS = [
        dict(name='gru64_1layer', hidden=64, layers=1, dropout=0.0,
             lr=1e-3, batch_size=256, max_epochs=12, patience=3, max_seconds=600),
        dict(name='gru96_2layer', hidden=96, layers=2, dropout=0.10,
             lr=1e-3, batch_size=256, max_epochs=12, patience=3, max_seconds=600),
        dict(name='gru128_2layer', hidden=128, layers=2, dropout=0.15,
             lr=7.5e-4, batch_size=256, max_epochs=12, patience=3, max_seconds=600),
    ]


    def seed_gru(seed=42):
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.set_num_threads(4)
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
        # Strict setting surfaces unsupported nondeterministic operations rather than hiding them.
        torch.use_deterministic_algorithms(True)


    class PressureGRU(nn.Module):
        def __init__(self, n_features, hidden=64, layers=1, dropout=0.0):
            super().__init__()
            self.gru = nn.GRU(n_features, hidden, num_layers=layers,
                              batch_first=True, dropout=dropout if layers > 1 else 0.0)
            self.head = nn.Sequential(nn.Linear(hidden, 32), nn.SiLU(), nn.Linear(32, 1))

        def forward(self, x):
            # No supplied hidden state: every batch/breath starts with zero state.
            sequence, _ = self.gru(x)
            return self.head(sequence).squeeze(-1)


    def inspiration_mae(y_true, y_pred, u_out):
        mask = np.asarray(u_out) == 0
        if not np.any(mask):
            raise ValueError('No inspiration points to score')
        errors = np.abs(np.asarray(y_pred, dtype=np.float64) -
                        np.asarray(y_true, dtype=np.float64))
        return float(errors[mask].sum(dtype=np.float64) / mask.sum())


    def per_rc_mae(y_true, y_pred, u_out, rc):
        """rc is [n_breaths, 2], from original unscaled R and C, for diagnostics only."""
        if rc is None:
            return []
        rc = np.asarray(rc)
        if rc.shape != (len(y_true), 2):
            raise ValueError('rc must have shape [n_breaths, 2]')
        result = []
        for r, c in sorted({tuple(row) for row in rc.tolist()}):
            rows = (rc[:, 0] == r) & (rc[:, 1] == c)
            mask = np.asarray(u_out)[rows] == 0
            result.append(dict(R=float(r), C=float(c), breaths=int(rows.sum()),
                               scored_points=int(mask.sum()),
                               mae=inspiration_mae(np.asarray(y_true)[rows],
                                                   np.asarray(y_pred)[rows],
                                                   np.asarray(u_out)[rows])))
        return result


    @torch.no_grad()
    def _predict_gru_scaled(model, x_standardized, batch_size=512):
        model.eval()
        device = next(model.parameters()).device
        output = np.empty(x_standardized.shape[:2], dtype=np.float32)
        for start in range(0, len(x_standardized), batch_size):
            stop = min(start + batch_size, len(x_standardized))
            batch = torch.from_numpy(np.ascontiguousarray(x_standardized[start:stop])).to(device)
            output[start:stop] = model(batch).cpu().numpy()
        return output


    def predict_gru(bundle, x, batch_size=512):
        """Returns all timesteps in original array order, in native pressure units."""
        x = np.asarray(x, dtype=np.float32)
        if x.ndim != 3 or x.shape[1:] != bundle['input_shape']:
            raise ValueError('Unexpected sequence/feature shape')
        if not np.isfinite(x).all():
            raise ValueError('Nonfinite GRU input')
        standardized = np.ascontiguousarray((x - bundle['x_mean']) / bundle['x_std'])
        scaled = _predict_gru_scaled(bundle['model'], standardized, batch_size)
        output = scaled * bundle['target_scale'] + bundle['target_center']
        if not np.isfinite(output).all():
            raise ValueError('Nonfinite GRU prediction')
        return output.astype(np.float32, copy=False)


    def fit_gru_trial(x_train, y_train, u_out_train, x_val, y_val, u_out_val,
                      config, seed=42, val_rc=None, checkpoint_path=None,
                      epoch_jsonl_path=None, on_epoch=None):
        """Input arrays: X=[breaths,80,features], y/u_out=[breaths,80].

        Supply ONLY the selected training breaths in x_train/y_train. Validation must
        contain disjoint whole breaths. No target pressure is included among X features.
        Scaling is fitted exclusively here on training inputs/targets; val_rc never fits.
        max_seconds is checked between epochs, so a trial may exceed it by one epoch.
        Returns (bundle, report, full_validation_predictions).
        """
        if not torch.cuda.is_available():
            raise RuntimeError('A CUDA GPU is required for these GRU trials')
        seed_gru(seed)
        config = dict(config)
        x_train = np.asarray(x_train, dtype=np.float32)
        x_val = np.asarray(x_val, dtype=np.float32)
        y_train = np.asarray(y_train, dtype=np.float32)
        y_val = np.asarray(y_val, dtype=np.float32)
        u_out_train = np.asarray(u_out_train)
        u_out_val = np.asarray(u_out_val)
        if x_train.ndim != 3 or x_train.shape[1] != 80 or x_val.shape[1:] != x_train.shape[1:]:
            raise ValueError('Expected compatible 80-timestep sequence arrays')
        for x, y, out in ((x_train, y_train, u_out_train), (x_val, y_val, u_out_val)):
            if y.shape != x.shape[:2] or out.shape != y.shape:
                raise ValueError('Input/target/mask shapes disagree')
            if not np.isfinite(x).all() or not np.isfinite(y).all():
                raise ValueError('Nonfinite training/validation values')
            if not np.isin(out, [0, 1]).all() or not np.any(out == 0):
                raise ValueError('Invalid inspiration mask')

        x_mean = x_train.mean(axis=(0, 1), dtype=np.float64).astype(np.float32)
        x_std = x_train.std(axis=(0, 1), dtype=np.float64).astype(np.float32)
        x_std[x_std < 1e-6] = 1.0
        train_z = np.ascontiguousarray((x_train - x_mean) / x_std)
        val_z = np.ascontiguousarray((x_val - x_mean) / x_std)
        train_mask = (u_out_train == 0).astype(np.float32)
        scored_targets = y_train[u_out_train == 0]
        center = float(np.median(scored_targets))
        q25, q75 = np.quantile(scored_targets, [0.25, 0.75])
        scale = max(float(q75 - q25), 1.0)
        train_y_z = np.ascontiguousarray((y_train - center) / scale)
        dataset = TensorDataset(torch.from_numpy(train_z), torch.from_numpy(train_y_z),
                                torch.from_numpy(train_mask))
        generator = torch.Generator().manual_seed(seed)
        loader = DataLoader(dataset, batch_size=config['batch_size'], shuffle=True,
                            num_workers=0, pin_memory=True, generator=generator)
        model = PressureGRU(x_train.shape[-1], config['hidden'], config['layers'],
                            config['dropout']).cuda()
        optimizer = torch.optim.Adam(model.parameters(), lr=config['lr'])
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min',
                                                             factor=0.5, patience=1, min_lr=config.get('min_lr',0.))
        best_mae, best_epoch, best_state = float('inf'), 0, None
        history, bad_epochs = [], 0
        started = time.monotonic()
        torch.cuda.reset_peak_memory_stats()
        for epoch in range(1, config['max_epochs'] + 1):
            epoch_started = time.monotonic()
            model.train()
            train_error, train_count = 0.0, 0.0
            for xb, yb, mask in loader:
                xb = xb.cuda(non_blocking=True)
                yb = yb.cuda(non_blocking=True)
                mask = mask.cuda(non_blocking=True)
                optimizer.zero_grad(set_to_none=True)
                pred = model(xb)
                absolute = (pred - yb).abs() * mask
                count = mask.sum()
                if count.item() == 0:
                    continue
                loss = absolute.sum() / count
                if not torch.isfinite(loss):
                    raise RuntimeError('Nonfinite GRU loss')
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
                train_error += float(absolute.detach().sum()) * scale
                train_count += float(count)
            val_predictions = _predict_gru_scaled(model, val_z) * scale + center
            val_mae = inspiration_mae(y_val, val_predictions, u_out_val)
            if val_mae < best_mae - 1e-6:
                best_mae, best_epoch, bad_epochs = val_mae, epoch, 0
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                if checkpoint_path is not None:
                    path = Path(checkpoint_path)
                    path.parent.mkdir(parents=True, exist_ok=True)
                    temporary = path.with_name(path.name + '.tmp')
                    torch.save(dict(state_dict=best_state, config=config, seed=seed,
                                    n_features=x_train.shape[-1], x_mean=x_mean, x_std=x_std,
                                    target_center=center, target_scale=scale,
                                    best_epoch=best_epoch, best_val_mae=best_mae), temporary)
                    os.replace(temporary, path)
            else:
                bad_epochs += 1
            scheduler.step(val_mae)
            record = dict(trial=config['name'], seed=seed, epoch=epoch,
                          train_inspiratory_mae=train_error / train_count,
                          validation_inspiratory_mae=val_mae, best_val_mae=best_mae,
                          best_epoch=best_epoch, lr=optimizer.param_groups[0]['lr'],
                          epoch_seconds=time.monotonic() - epoch_started,
                          elapsed_seconds=time.monotonic() - started,
                          peak_gpu_memory_MiB=torch.cuda.max_memory_allocated() / 2**20)
            history.append(record)
            if epoch_jsonl_path is not None:
                path = Path(epoch_jsonl_path)
                path.parent.mkdir(parents=True, exist_ok=True)
                with path.open('a') as stream:
                    stream.write(json.dumps(record) + '\n')
            if on_epoch is not None:
                on_epoch(record)
            if bad_epochs >= config['patience'] or record['elapsed_seconds'] >= config['max_seconds']:
                break
        if best_state is None:
            raise RuntimeError('No valid GRU checkpoint produced')
        model.load_state_dict(best_state)
        model.eval()
        bundle = dict(model=model, config=config, seed=seed, x_mean=x_mean, x_std=x_std,
                      target_center=center, target_scale=scale, input_shape=x_train.shape[1:])
        val_predictions = predict_gru(bundle, x_val)
        restored_mae = inspiration_mae(y_val, val_predictions, u_out_val)
        if not np.isclose(restored_mae, best_mae, rtol=1e-6, atol=1e-6):
            raise RuntimeError('Restored checkpoint MAE differs from logged best')
        report = dict(config=config, seed=seed, best_epoch=best_epoch,
                      validation_inspiratory_mae=restored_mae,
                      elapsed_seconds=time.monotonic() - started,
                      train_breaths=len(x_train), val_breaths=len(x_val),
                      scored_val_points=int((u_out_val == 0).sum()),
                      per_rc=per_rc_mae(y_val, val_predictions, u_out_val, val_rc),
                      epochs=history)
        return bundle, report, val_predictions


    def predictions_by_rc(predictions, rc):
        """Optional raw slices; keep these arrays outside JSON reports."""
        rc = np.asarray(rc)
        predictions = np.asarray(predictions)
        if rc.shape != (len(predictions), 2):
            raise ValueError('rc must have shape [n_breaths,2]')
        result = {}
        for r, c in sorted({tuple(row) for row in rc.tolist()}):
            rows = np.flatnonzero((rc[:, 0] == r) & (rc[:, 1] == c))
            result[(r, c)] = dict(breath_rows=rows, predictions=predictions[rows])
        return result

In [6]:
def frozen_budget(result):
    if result['definition']['family']=='LightGBM':
        return dict(kind='trees',iterations=int(result['best_iteration']),selected_using='A only')
    history=json.loads(Path(result['epoch_log_file']).read_text())['epochs']
    epochs=int(result['best_epoch'])
    rates=[result['definition']['params']['lr']]+[r['lr'] for r in history[:epochs-1]]
    assert len(rates)==epochs
    return dict(kind='GRU',epochs=epochs,learning_rates_by_epoch=rates,selected_using='A only')

def fit_gru_without_validation(x_train,y_train,u_out_train,config,seed,budget,checkpoint,epoch_log):
    # No B validation inputs or labels are accepted by this function.
    seed_gru(seed)
    mean=x_train.mean(axis=(0,1),dtype=np.float64).astype(np.float32)
    std=x_train.std(axis=(0,1),dtype=np.float64).astype(np.float32);std[std<1e-6]=1.
    z=np.ascontiguousarray((x_train-mean)/std)
    scored=y_train[u_out_train==0];center=float(np.median(scored))
    q25,q75=np.quantile(scored,[.25,.75]);scale=max(float(q75-q25),1.)
    yz=np.ascontiguousarray((y_train-center)/scale)
    mask=(u_out_train==0).astype(np.float32)
    dataset=TensorDataset(torch.from_numpy(z),torch.from_numpy(yz),torch.from_numpy(mask))
    loader=DataLoader(dataset,batch_size=config['batch_size'],shuffle=True,num_workers=0,
                      pin_memory=True,generator=torch.Generator().manual_seed(seed))
    model=PressureGRU(x_train.shape[-1],config['hidden'],config['layers'],config['dropout']).cuda()
    optimizer=torch.optim.Adam(model.parameters(),lr=config['lr'])
    history=[];started=time.monotonic()
    for epoch,rate in enumerate(budget['learning_rates_by_epoch'],1):
        for group in optimizer.param_groups:group['lr']=rate
        model.train();error=0.;count=0.
        for xb,yb,mb in loader:
            xb=xb.cuda(non_blocking=True);yb=yb.cuda(non_blocking=True);mb=mb.cuda(non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            absolute=(model(xb)-yb).abs()*mb
            n=mb.sum();loss=absolute.sum()/n
            if not torch.isfinite(loss):raise RuntimeError('Nonfinite fixed-budget loss')
            loss.backward();nn.utils.clip_grad_norm_(model.parameters(),1.);optimizer.step()
            error+=float(absolute.detach().sum())*scale;count+=float(n)
        record=dict(epoch=epoch,lr=rate,train_inspiratory_mae=error/count,
                    elapsed_seconds=time.monotonic()-started,validation_used=False)
        history.append(record)
        append_event('B_fixed_GRU_epoch',trial=config['name'],**record)
    state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
    torch.save(dict(state_dict=state,config=config,seed=seed,n_features=x_train.shape[-1],
        x_mean=mean,x_std=std,target_center=center,target_scale=scale,
        best_epoch=budget['epochs'],best_val_mae=None,frozen_budget=budget),checkpoint)
    model.eval()
    bundle=dict(model=model,x_mean=mean,x_std=std,target_center=center,target_scale=scale,input_shape=x_train.shape[1:])
    report=dict(epochs=history,termination='fixed_A_budget',validation_used=False,frozen_budget=budget,
                elapsed_seconds=time.monotonic()-started)
    json_write(epoch_log,report)
    return bundle,report

In [7]:
def canonical_hash(value):
    return hashlib.sha256(json.dumps(value,sort_keys=True).encode()).hexdigest()

def result_path(trial): return OUT/(trial+'.json')

def read_result(trial): return json.loads(result_path(trial).read_text())

def rows_for(split):
    indices=VALS[split]
    mask=UOUT[indices]==0
    labels=Y[indices][mask]
    rc=np.repeat(RC[indices],80,axis=0)[mask.ravel()]
    ids=ROW_IDS[indices][mask]
    return labels,rc,ids

def score_prediction(trial,prediction,split):
    target,rc,ids=rows_for(split)
    prediction=np.asarray(prediction,dtype=np.float64)
    assert prediction.shape==target.shape and np.isfinite(prediction).all()
    error=np.abs(prediction-target)
    np.save(OUT/(trial+'_validation_predictions.npy'),prediction)
    summary=pd.DataFrame({'R':rc[:,0],'C':rc[:,1],'absolute_error':error})
    summary.groupby(['R','C']).absolute_error.agg(['mean','count']).to_csv(OUT/(trial+'_by_rc.csv'))
    return float(error.mean()),len(target)

def refresh_table():
    with IO_LOCK:
        result=[]
        for p in OUT.glob('*.json'):
            r=json.loads(p.read_text())
            if isinstance(r,dict) and r.get('record_type')=='trial': result.append(r)
        pd.DataFrame(result).to_csv(OUT/'results.csv.tmp',index=False)
        os.replace(OUT/'results.csv.tmp',OUT/'results.csv')

def load_gru_bundle(path):
    saved=torch.load(path,map_location='cpu')
    cfg=saved['config']
    model=PressureGRU(saved['n_features'],cfg['hidden'],cfg['layers'],cfg['dropout']).cuda()
    model.load_state_dict(saved['state_dict']);model.eval()
    return dict(model=model,x_mean=saved['x_mean'],x_std=saved['x_std'],
        target_center=saved['target_center'],target_scale=saved['target_scale'],
        input_shape=(80,saved['n_features']))

def predict_saved(result,sequence):
    definition=result['definition']
    z,used=transform(sequence,definition)
    assert used==result['input_columns']
    if definition['family']=='LightGBM':
        model=lgb.Booster(model_file=result['model_file'])
        output=model.predict(z.reshape(-1,len(used)),num_threads=THREADS).reshape(z.shape[:2])
    else:
        bundle=load_gru_bundle(result['model_file'])
        output=predict_gru(bundle,z)
        del bundle;torch.cuda.empty_cache()
    assert output.shape==sequence.shape[:2] and np.isfinite(output).all()
    return output

def fit_trial(trial,definition,fit_group='screen',split='A',seed=MODEL_SEEDS[0],reuse=None,budget=None):
    fit_ids=FITS[fit_group];val_ids=VALS[split]
    assert not np.intersect1d(fit_ids,val_ids).size
    assert not np.intersect1d(fit_ids,a_val).size
    assert (split=='B')==(budget is not None), 'B requires an A-frozen budget; A uses its own checkpoint selection'
    identity=dict(definition=definition,seed=seed,fit_group=fit_group,split=split,frozen_budget=budget,
        fit_breath_sha256=hashlib.sha256(breath_ids[fit_ids].tobytes()).hexdigest(),
        val_breath_sha256=hashlib.sha256(breath_ids[val_ids].tobytes()).hexdigest(),
        design_sha256=canonical_hash(fingerprint),reuse=str(reuse) if reuse else None)
    identity_hash=canonical_hash(identity)
    if result_path(trial).exists():
        existing=read_result(trial)
        assert existing['identity_sha256']==identity_hash, 'Trial identity mismatch'
        assert Path(existing['model_file']).is_file()
        assert digest(Path(existing['model_file']))==existing['model_sha256']
        assert (OUT/(trial+'_validation_predictions.npy')).is_file()
        print('REUSE completed',trial,flush=True)
        return existing
    append_event('trial_started',trial=trial,identity=identity)
    print('START',trial,definition,fit_group,split,seed,flush=True)
    begin=time.monotonic();fit_seconds=0.;curves={};bundle=None;model=None
    diagnostic=train_diagnostic_indices(fit_ids)
    checkpoint=Path(reuse) if reuse else OUT/(trial+('.txt' if definition['family']=='LightGBM' else '.pt'))
    try:
        zval,used=transform(X[val_ids],definition)
        zdia,_=transform(X[diagnostic],definition)
        if definition['family']=='LightGBM':
            mask=UOUT[val_ids]==0
            valx=zval[mask];valy=Y[val_ids][mask]
            if reuse:
                model=lgb.Booster(model_file=str(checkpoint))
                best_iteration=model.num_trees();termination='reused_prior_checkpoint'
            else:
                trainx,trainy,train_names=flat_scored(fit_ids,definition)
                assert used==train_names
                params=dict(definition['params'])
                if budget is not None:params['n_estimators']=budget['iterations']
                learner=lgb.LGBMRegressor(**params,objective='regression_l1',
                    subsample=.9,subsample_freq=1,colsample_bytree=.9,n_jobs=THREADS,
                    random_state=seed,deterministic=True,force_col_wise=True,verbosity=-1)
                cat=[used.index('R'),used.index('C')] if definition['rc']=='categorical' else 'auto'
                started=time.monotonic()
                if budget is None:
                    learner.fit(trainx,trainy,feature_name=used,categorical_feature=cat,
                        eval_set=[(valx,valy)],eval_names=['validation'],eval_metric='l1',
                        callbacks=[lgb.early_stopping(75,first_metric_only=True,verbose=False),
                                   lgb.record_evaluation(curves),lgb.log_evaluation(500)])
                else:
                    learner.fit(trainx,trainy,feature_name=used,categorical_feature=cat)
                fit_seconds=time.monotonic()-started
                best_iteration=int(learner.best_iteration_) if budget is None else int(budget['iterations'])
                model=learner.booster_
                model.save_model(str(checkpoint),num_iteration=best_iteration)
                # Reload the serialized best checkpoint for every reported prediction.
                model=lgb.Booster(model_file=str(checkpoint))
                termination=('fixed_A_budget' if budget is not None else
                    'tree_cap' if len(curves['validation']['l1'])>=definition['params']['n_estimators'] else 'early_stopping')
                json_write(OUT/(trial+'_curve.json'),curves)
                del trainx,trainy,learner;gc.collect()
            pred=model.predict(valx,num_threads=THREADS)
            dia_mask=UOUT[diagnostic]==0
            dia_pred=model.predict(zdia[dia_mask],num_threads=THREADS)
            train_mae=float(np.abs(dia_pred-Y[diagnostic][dia_mask]).mean())
            best_epoch=None
        else:
            assert definition['rc']=='numeric'
            if reuse:
                bundle=load_gru_bundle(checkpoint)
                saved=torch.load(checkpoint,map_location='cpu')
                best_epoch=int(saved['best_epoch']);termination='reused_prior_checkpoint'
                pred_sequence=predict_gru(bundle,zval)
            else:
                zfit,_=transform(X[fit_ids],definition)
                cfg=dict(definition['params'],name=trial)
                if budget is None:
                    bundle,report,pred_sequence=fit_gru_trial(zfit,Y[fit_ids],UOUT[fit_ids],
                        zval,Y[val_ids],UOUT[val_ids],cfg,seed=seed,val_rc=RC[val_ids],
                        checkpoint_path=checkpoint,epoch_jsonl_path=OUT/'epochs.jsonl')
                else:
                    bundle,report=fit_gru_without_validation(zfit,Y[fit_ids],UOUT[fit_ids],cfg,seed,
                        budget,checkpoint,OUT/(trial+'_epochs.json'))
                    pred_sequence=predict_gru(bundle,zval)
                fit_seconds=report['elapsed_seconds'];best_epoch=int(report['best_epoch']) if budget is None else int(budget['epochs'])
                final=report['epochs'][-1]
                termination=('fixed_A_budget' if budget is not None else
                    'early_stopping' if final['epoch']-best_epoch>=cfg['patience'] else
                    'time_cap' if final['elapsed_seconds']>=cfg['max_seconds'] else 'epoch_cap')
                report['termination']=termination
                json_write(OUT/(trial+'_epochs.json'),report)
                del zfit;gc.collect()
            pred=pred_sequence[UOUT[val_ids]==0]
            dia_pred=predict_gru(bundle,zdia)
            train_mae=inspiration_mae(Y[diagnostic],dia_pred,UOUT[diagnostic])
            best_iteration=None
        mae,scored=score_prediction(trial,pred,split)
        resolved=(model.params if definition['family']=='LightGBM' else dict(definition['params'],
            optimizer='Adam',Adam_betas=[.9,.999],Adam_eps=1e-8,Adam_weight_decay=0.,
            scheduler='ReduceLROnPlateau' if budget is None else 'replayed_A_schedule',scheduler_factor=.5,scheduler_patience=1,
            gradient_clip_norm=1.,input_scaling='training mean/std',target_scaling='training inspiration median/IQR'))
        result=dict(record_type='trial',trial=trial,definition=definition,resolved_parameters=resolved,identity=identity,
            identity_sha256=identity_hash,seed=seed,split=split,fit_group=fit_group,
            training_breaths=len(fit_ids),validation_breaths=len(val_ids),scored_points=scored,
            validation_mae=mae,train_diagnostic_mae=train_mae,train_diagnostic_breaths=len(diagnostic),
            fit_seconds=fit_seconds,runtime_seconds=time.monotonic()-begin,reused_checkpoint=bool(reuse),
            best_iteration=best_iteration,best_epoch=best_epoch,termination=termination,frozen_budget=budget,
            epoch_log_file=(str(PREV/(checkpoint.stem+'_report.json')) if reuse else str(OUT/(trial+'_epochs.json'))) if definition['family']=='GRU' else None,
            model_file=str(checkpoint),model_sha256=digest(checkpoint),input_columns=used,
            categorical_columns=['R','C'] if definition['rc']=='categorical' else [],
            category_mappings={'R':[5,20,50],'C':[10,20,50]} if definition['rc']=='categorical' else {},
            prediction_file=str(OUT/(trial+'_validation_predictions.npy')),
            status='complete')
        json_write(result_path(trial),result)
        append_event('trial_completed',trial=trial,validation_mae=mae,fit_seconds=fit_seconds,
            runtime_seconds=result['runtime_seconds'],termination=termination)
        refresh_table()
        print('COMPLETE',trial,mae,'train diagnostic',train_mae,'seconds',result['runtime_seconds'],flush=True)
        return result
    except Exception as error:
        append_event('trial_failed',trial=trial,error=repr(error),elapsed_seconds=time.monotonic()-begin)
        raise
    finally:
        del model,bundle
        gc.collect()
        if definition['family']=='GRU' and 'torch' in globals(): torch.cuda.empty_cache()

In [8]:
if RUN_EXPERIMENTS:
    tree_anchor = fit_trial('screen_tree_reference', TREE_REFERENCE, reuse=PREV / 'lgb_2500_255.txt')
    incumbent = fit_trial('full_incumbent', TREE_REFERENCE, fit_group='full', reuse=PREV / 'lgb_2500_255_fullpool.txt')
    assert np.isclose(tree_anchor['validation_mae'], 0.7180270039640434, atol=1e-07)
    assert np.isclose(incumbent['validation_mae'], 0.664386093036891, atol=1e-07)

START screen_tree_reference {'family': 'LightGBM', 'params': {'n_estimators': 2500, 'num_leaves': 255, 'learning_rate': 0.04, 'min_child_samples': 20, 'max_bin': 255, 'reg_lambda': 2.0}, 'history': 'both', 'rc': 'numeric'} screen A 20261102
COMPLETE screen_tree_reference 0.7180270039640434 train diagnostic 0.5693059039689701 seconds 20.409536564722657
START full_incumbent {'family': 'LightGBM', 'params': {'n_estimators': 2500, 'num_leaves': 255, 'learning_rate': 0.04, 'min_child_samples': 20, 'max_bin': 255, 'reg_lambda': 2.0}, 'history': 'both', 'rc': 'numeric'} full A 20261102
COMPLETE full_incumbent 0.664386093036891 train diagnostic 0.5844564741037677 seconds 20.92025898769498


In [9]:
if RUN_EXPERIMENTS:
    tree_pool=ThreadPoolExecutor(max_workers=2)
    tree_futures=[]

In [10]:
if RUN_EXPERIMENTS:
    tree_futures.append(tree_pool.submit(fit_trial,'screen_'+TREE_SCREEN[0][0],TREE_SCREEN[0][1]))
    print('Queued tree arm:',TREE_SCREEN[0][0])

Queued tree arm: history_neither


In [11]:
if RUN_EXPERIMENTS:
    tree_futures.append(tree_pool.submit(fit_trial,'screen_'+TREE_SCREEN[1][0],TREE_SCREEN[1][1]))
    print('Queued tree arm:',TREE_SCREEN[1][0])

Queued tree arm: history_lags


In [12]:
if RUN_EXPERIMENTS:
    tree_futures.append(tree_pool.submit(fit_trial,'screen_'+TREE_SCREEN[2][0],TREE_SCREEN[2][1]))
    print('Queued tree arm:',TREE_SCREEN[2][0])

START screen_history_neither {'family': 'LightGBM', 'params': {'n_estimators': 2500, 'num_leaves': 255, 'learning_rate': 0.04, 'min_child_samples': 20, 'max_bin': 255, 'reg_lambda': 2.0}, 'history': 'neither', 'rc': 'numeric'} screen A 20261102
Queued tree arm: history_changes


In [13]:
if RUN_EXPERIMENTS:
    tree_futures.append(tree_pool.submit(fit_trial,'screen_'+TREE_SCREEN[3][0],TREE_SCREEN[3][1]))
    print('Queued tree arm:',TREE_SCREEN[3][0])

START screen_history_lags {'family': 'LightGBM', 'params': {'n_estimators': 2500, 'num_leaves': 255, 'learning_rate': 0.04, 'min_child_samples': 20, 'max_bin': 255, 'reg_lambda': 2.0}, 'history': 'lags', 'rc': 'numeric'} screen A 20261102
Queued tree arm: rc_categorical


In [14]:
if RUN_EXPERIMENTS:
    tree_futures.append(tree_pool.submit(fit_trial,'screen_'+TREE_SCREEN[4][0],TREE_SCREEN[4][1]))
    print('Queued tree arm:',TREE_SCREEN[4][0])

Queued tree arm: lr_002


In [15]:
if RUN_EXPERIMENTS:
    tree_futures.append(tree_pool.submit(fit_trial,'screen_'+TREE_SCREEN[5][0],TREE_SCREEN[5][1]))
    print('Queued tree arm:',TREE_SCREEN[5][0])

Queued tree arm: lr_008


In [16]:
if RUN_EXPERIMENTS:
    tree_futures.append(tree_pool.submit(fit_trial,'screen_'+TREE_SCREEN[6][0],TREE_SCREEN[6][1]))
    print('Queued tree arm:',TREE_SCREEN[6][0])

Queued tree arm: trees_5000


In [17]:
if RUN_EXPERIMENTS:
    tree_futures.append(tree_pool.submit(fit_trial,'screen_'+TREE_SCREEN[7][0],TREE_SCREEN[7][1]))
    print('Queued tree arm:',TREE_SCREEN[7][0])

Queued tree arm: leaves_127


In [18]:
if RUN_EXPERIMENTS:
    tree_futures.append(tree_pool.submit(fit_trial,'screen_'+TREE_SCREEN[8][0],TREE_SCREEN[8][1]))
    print('Queued tree arm:',TREE_SCREEN[8][0])

Queued tree arm: leaves_511


In [19]:
if RUN_EXPERIMENTS:
    tree_futures.append(tree_pool.submit(fit_trial,'screen_'+TREE_SCREEN[9][0],TREE_SCREEN[9][1]))
    print('Queued tree arm:',TREE_SCREEN[9][0])

Queued tree arm: l2_10


In [20]:
if RUN_EXPERIMENTS:
    try:
        for future in tree_futures: future.result()
    finally:
        tree_pool.shutdown(wait=True)
    print('All CPU screening arms completed; GRUs deferred.')

[500]	validation's l1: 0.992943
[500]	validation's l1: 0.855187
[1000]	validation's l1: 0.939447
[1000]	validation's l1: 0.79826
[1500]	validation's l1: 0.910183
[1500]	validation's l1: 0.77259
[2000]	validation's l1: 0.893202
[2000]	validation's l1: 0.756422
[2500]	validation's l1: 0.88131
[2500]	validation's l1: 0.745866
COMPLETE screen_history_neither 0.8813103567248741 train diagnostic 0.7295054138892936 seconds 104.99293706379831
START screen_history_changes {'family': 'LightGBM', 'params': {'n_estimators': 2500, 'num_leaves': 255, 'learning_rate': 0.04, 'min_child_samples': 20, 'max_bin': 255, 'reg_lambda': 2.0}, 'history': 'changes', 'rc': 'numeric'} screen A 20261102
COMPLETE screen_history_lags 0.745866348274679 train diagnostic 0.5970311444611343 seconds 115.35873611271381
START screen_rc_categorical {'family': 'LightGBM', 'params': {'n_estimators': 2500, 'num_leaves': 255, 'learning_rate': 0.04, 'min_child_samples': 20, 'max_bin': 255, 'reg_lambda': 2.0}, 'history': 'both', 